<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">

# **NOTEBOOK 15**
---



# **Modelos del lenguaje basados en redes neuronales artificiales**

## **Modelos seq2seq 2**

Cuando la entrada de cada paso del decodificador proviene de la salida del paso anterior, estamos hablando de un modelo Seq2Seq con realimentación (feedback). Esto es especialmente común en tareas como la generación de texto.

1. **Inicio de la Secuencia**: Se inicia la generación con un token especial, como `<SOS>` (Start of Sequence).

2. **Generación Paso a Paso**:
   - En el primer paso, el decodificador recibe el `<SOS>` y el estado del codificador como entrada.
   - El decodificador procesa esta entrada y genera una salida para este paso.
   - La salida generada se convierte en la entrada para el siguiente paso, junto con el estado actualizado del decodificador.
   - Este proceso se repite hasta que se genera un token especial de fin de secuencia (`<EOS>`, End of Sequence) o hasta alcanzar un límite máximo de longitud.

3. **Ventajas y Desventajas**:
   - **Ventajas**: Esta forma de generar secuencias puede ayudar a mantener la coherencia en las secuencias generadas, ya que cada nueva palabra o token tiene en cuenta lo que ya se ha generado.
   - **Desventajas**: Puede ser más lento, ya que cada paso depende del anterior, y errores en un paso pueden propagarse y afectar los pasos siguientes.


Imagina que tienes un modelo seq2seq entrenado para traducir inglés a español. Para la frase "How are you?", el proceso sería algo así:

1. El codificador procesa "How are you?" y genera un vector latente.
2. El decodificador recibe el vector latente y el token `<SOS>`.
3. El decodificador genera "¿Cómo", actualiza su estado y toma "¿Cómo" como entrada para el siguiente paso.
4. El decodificador genera "estás", actualiza su estado y toma "estás" como entrada para el siguiente paso.
5. Y así sucesivamente, hasta generar `<EOS>` para indicar el final de la secuencia.

Este modelo de generación permite que el decodificador tenga en cuenta no solo el contexto proporcionado por el codificador, sino también la estructura de la secuencia que está generando, paso a paso.


<p align="center">
<img src="imgs/seq2seq_feedback.svg" width="70%">
</p>


### **Teacher Forcing**

**Teacher forcing** es una técnica utilizada en el entrenamiento de modelos seq2seq en la que, en un porcentaje de las veces, se utiliza la salida real (etiqueta) de un paso de tiempo como entrada para el siguiente paso, en lugar de la salida predicha por el modelo. Esta técnica puede ayudar a acelerar la convergencia y mejorar el rendimiento del modelo, especialmente en las etapas iniciales del entrenamiento.

#### **¿Cómo funciona?**

1. **Durante el Entrenamiento**: En cada paso de tiempo y con una cierta probabilidad de que suceda, en lugar de pasar la predicción del modelo del paso anterior como entrada al siguiente paso, se pasa la palabra real de la secuencia objetivo. Esto proporciona al modelo información directa y clara sobre cómo debería haber respondido en el paso anterior, independientemente de si la predicción fue correcta o no.

2. **Durante la Evaluación/Predicción**: El modelo debe generar secuencias por sí mismo, utilizando sus propias predicciones del paso anterior para el siguiente paso. Durante esta fase, no se utiliza "teacher forcing".

#### **Ventajas de Teacher Forcing:**

1. **Aprendizaje más Rápido**: Al proporcionar al modelo la respuesta correcta en cada paso, se reduce la propagación de errores a través de la secuencia, lo que puede llevar a un aprendizaje más rápido.

2. **Mejor Rendimiento**: Puede resultar en un mejor rendimiento del modelo, especialmente en las primeras etapas del entrenamiento.

### **Implementación traductor inglés a español**

#### **Dataset Europarl**

El conjunto de datos Europarl contiene las transcripciones de los procedimientos del Parlamento Europeo, proporcionando una valiosa fuente de textos paralelos en 21 idiomas europeos. Las oraciones están alineadas entre los idiomas, lo que lo hace especialmente útil para tareas de traducción automática. 

Descargamos el dataset Europarl para español-inglés. Una vez descargado tendremos dos ficheros de texto, uno para cada idioma con las frases alineadas. El código siguiente muestra las primeras frases de cada fichero.

In [2]:
def read_translation(archivo_ingles, archivo_espanol):
    with open(archivo_ingles, 'r', encoding='utf-8') as f_ingles, open(archivo_espanol, 'r', encoding='utf-8') as f_espanol:
        for oracion_ingles, oracion_espanol in zip(f_ingles, f_espanol):
            yield oracion_ingles.strip(), oracion_espanol.strip()


archivo_ingles = 'data/europarl/europarl-v7.es-en.en'
archivo_espanol = 'data/europarl/europarl-v7.es-en.es'

# Leer el conjunto de datos
for i, (ingles, espanol) in enumerate(read_translation(archivo_ingles, archivo_espanol)):
    print('Inglés:', ingles)
    print('Español:', espanol)
    print('---')
    if i == 15:
        break

Inglés: Resumption of the session
Español: Reanudación del período de sesiones
---
Inglés: I declare resumed the session of the European Parliament adjourned on Friday 17 December 1999, and I would like once again to wish you a happy new year in the hope that you enjoyed a pleasant festive period.
Español: Declaro reanudado el período de sesiones del Parlamento Europeo, interrumpido el viernes 17 de diciembre pasado, y reitero a Sus Señorías mi deseo de que hayan tenido unas buenas vacaciones.
---
Inglés: Although, as you will have seen, the dreaded 'millennium bug' failed to materialise, still the people in a number of countries suffered a series of natural disasters that truly were dreadful.
Español: Como todos han podido comprobar, el gran "efecto del año 2000" no se ha producido. En cambio, los ciudadanos de varios de nuestros países han sido víctimas de catástrofes naturales verdaderamente terribles.
---
Inglés: You have requested a debate on this subject in the course of the next

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import zip_longest
import math
import random
import re
import time

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# CPU también funciona. Las GPU NVIDIA se usan si están disponibles.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

### **Datos**

Leemos pares alineados de Europarl. Para una primera práctica limitamos la longitud y el número de ejemplos. El subconjunto son los primeros pares válidos del fichero; no es una muestra representativa de todo Europarl. Se pueden ampliar estos límites después.

Eliminamos pares duplicados y asignamos cada frase inglesa a una única partición. Así, una misma entrada no aparece en entrenamiento y prueba, aunque tenga varias traducciones. Esto no elimina paráfrasis ni repeticiones entre documentos; una evaluación más exigente separaría debates completos.

La tokenización separa palabras y signos de puntuación. Es deliberadamente sencilla; no utiliza subpalabras. Las palabras que no estén en el vocabulario se representarán mediante `<unk>`.

In [ ]:
SOURCE_FILE = Path("data/europarl/europarl-v7.es-en.en")
TARGET_FILE = Path("data/europarl/europarl-v7.es-en.es")
MAX_WORDS = 30             # Sin contar los tokens especiales.
MAX_PAIRS = 30_000         # Usa None para conservar todos los pares válidos.
MIN_FREQUENCY = 2
MAX_VOCAB_SIZE = 12_000    # Incluye los cuatro tokens especiales.


def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)


def read_pairs(source_file, target_file, max_words, max_pairs):
    pairs = []
    seen = set()
    with source_file.open(encoding="utf-8") as source, target_file.open(encoding="utf-8") as target:
        for line_number, (en, es) in enumerate(zip_longest(source, target), start=1):
            if en is None or es is None:
                raise ValueError(f"Ficheros desalineados en la línea {line_number}")
            en_tokens, es_tokens = tokenize(en), tokenize(es)
            if not (1 <= len(en_tokens) <= max_words and 1 <= len(es_tokens) <= max_words):
                continue
            pair = (tuple(en_tokens), tuple(es_tokens))
            if pair in seen:
                continue
            seen.add(pair)
            pairs.append(pair)
            if max_pairs is not None and len(pairs) >= max_pairs:
                break
    return pairs


def split_pairs(pairs, seed):
    sources = sorted({en for en, es in pairs})
    if len(sources) < 20:
        raise ValueError("Se necesitan al menos 20 entradas distintas para esta práctica.")
    random.Random(seed).shuffle(sources)
    n_valid = max(1, int(0.1 * len(sources)))
    n_test = max(1, int(0.1 * len(sources)))
    valid_sources = set(sources[:n_valid])
    test_sources = set(sources[n_valid:n_valid + n_test])
    train, valid, test = [], [], []
    for pair in pairs:
        en, es = pair
        if en in valid_sources:
            valid.append(pair)
        elif en in test_sources:
            test.append(pair)
        else:
            train.append(pair)
    return train, valid, test


pairs = read_pairs(SOURCE_FILE, TARGET_FILE, MAX_WORDS, MAX_PAIRS)
train_pairs, valid_pairs, test_pairs = split_pairs(pairs, SEED)
print("Pares de entrenamiento, validación y prueba:",
      len(train_pairs), len(valid_pairs), len(test_pairs))
print("Ejemplo:", train_pairs[0])

### **Vocabulario**

Los identificadores especiales son iguales en ambos idiomas, pero cada idioma tiene su propio vocabulario y su propia capa de embeddings.

- `<pad>` rellena las secuencias de un lote.
- `<sos>` inicia la entrada del decoder.
- `<eos>` marca el final que debe aprender a predecir.
- `<unk>` representa palabras desconocidas.

No añadimos un `<pad>` extra a cada frase. El relleno se añade únicamente al formar el lote.

In [ ]:
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]


class Vocabulary:
    def __init__(self, sentences, min_frequency, max_size):
        counts = Counter(token for sentence in sentences for token in sentence)
        words = [word for word, count in counts.most_common()
                 if count >= min_frequency and word not in SPECIAL_TOKENS]
        self.itos = SPECIAL_TOKENS + words[:max_size - len(SPECIAL_TOKENS)]
        self.stoi = {word: index for index, word in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(token, UNK) for token in tokens]

    def decode(self, ids):
        words = []
        for index in ids:
            if index == EOS:
                break
            if index not in (PAD, SOS):
                words.append(self.itos[index])
        return words


src_vocab = Vocabulary((en for en, es in train_pairs), MIN_FREQUENCY, MAX_VOCAB_SIZE)
tgt_vocab = Vocabulary((es for en, es in train_pairs), MIN_FREQUENCY, MAX_VOCAB_SIZE)


class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.examples = []
        for en, es in pairs:
            source = torch.tensor(src_vocab.encode(en) + [EOS], dtype=torch.long)
            target = torch.tensor([SOS] + tgt_vocab.encode(es) + [EOS], dtype=torch.long)
            self.examples.append((source, target))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate_batch(examples):
    sources, targets = zip(*examples)
    source_lengths = torch.tensor([len(source) for source in sources])
    sources = pad_sequence(sources, batch_first=True, padding_value=PAD)
    targets = pad_sequence(targets, batch_first=True, padding_value=PAD)
    return sources, source_lengths, targets


BATCH_SIZE = 32
train_dataset = TranslationDataset(train_pairs, src_vocab, tgt_vocab)
valid_dataset = TranslationDataset(valid_pairs, src_vocab, tgt_vocab)
test_dataset = TranslationDataset(test_pairs, src_vocab, tgt_vocab)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_batch,
                          generator=torch.Generator().manual_seed(SEED))
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_batch)

print("Vocabularios inglés/español:", len(src_vocab), len(tgt_vocab))
sources, source_lengths, targets = next(iter(train_loader))
print("Entrada:", sources.shape, "Longitudes:", source_lengths.shape)
print("Objetivo con <sos> y <eos>:", targets.shape)

def unknown_rate(pairs, vocab, language_index):
    tokens = [token for pair in pairs for token in pair[language_index]]
    return sum(token not in vocab.stoi for token in tokens) / len(tokens)

print(f"Palabras desconocidas en prueba (EN): {unknown_rate(test_pairs, src_vocab, 0):.1%}")
print(f"Palabras desconocidas en prueba (ES): {unknown_rate(test_pairs, tgt_vocab, 1):.1%}")


### **Modelo**

##### **Encoder**

`pack_padded_sequence` indica las longitudes reales: el encoder no debe actualizar sus estados con los pasos de relleno.

Para una LSTM unidireccional, `hidden` y `cell` tienen forma `(n_capas, lote, dimensión_oculta)`. Pasamos **todas las capas** al decoder, que tiene la misma configuración.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                            num_layers=num_layers, batch_first=True)

    def forward(self, source, source_lengths):
        embedded = self.embedding(source)
        packed = pack_padded_sequence(embedded, source_lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, (hidden, cell) = self.lstm(packed)
        return hidden, cell

##### **Decoder**

La entrada es un identificador por ejemplo, de forma `(lote,)`. La capa de embeddings produce su vector y añadimos una dimensión temporal de longitud uno.

La salida son **logits**, no probabilidades. `CrossEntropyLoss` aplica internamente la transformación necesaria. Para elegir la siguiente palabra basta con `argmax` sobre los logits.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                            num_layers=num_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids, hidden, cell):
        embedded = self.embedding(token_ids).unsqueeze(1)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        logits = self.output_layer(output[:, 0, :])
        return logits, hidden, cell

##### **Seq2seq**

Para el objetivo `<sos> mi perro <eos>`, el primer paso recibe `<sos>` y debe predecir `mi`. Después recibe `mi` (teacher forcing) o su propia predicción. Nunca recibe la palabra que está intentando predecir en ese mismo paso.

La decisión de teacher forcing se toma por paso para todo el lote. En la rama sin teacher forcing realimentamos el **identificador predicho**, que pasa por la misma capa de embeddings utilizada con los tokens reales. `argmax` no es diferenciable; la pérdida de cada paso sí entrena sus logits y los estados recurrentes.

Durante el entrenamiento recorremos la longitud del objetivo del lote; los `<pad>` no contribuyen a la pérdida. La generación libre tendrá un método separado y no recibirá objetivos.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, source_lengths, target, teacher_forcing_ratio=0.5):
        hidden, cell = self.encoder(source, source_lengths)
        token_ids = target[:, 0]  # <sos>
        outputs = []

        for step in range(1, target.size(1)):
            logits, hidden, cell = self.decoder(token_ids, hidden, cell)
            outputs.append(logits)
            if random.random() < teacher_forcing_ratio:
                token_ids = target[:, step]
            else:
                token_ids = logits.argmax(dim=-1)

        # (lote, longitud_objetivo_sin_sos, vocabulario_español)
        return torch.stack(outputs, dim=1)

    @torch.no_grad()
    def generate(self, source, source_lengths, max_new_tokens=50):
        # El llamador debe activar model.eval(). No se utiliza la traducción real.
        hidden, cell = self.encoder(source, source_lengths)
        batch_size = source.size(0)
        token_ids = torch.full((batch_size,), SOS, dtype=torch.long, device=source.device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=source.device)
        predictions = []

        for _ in range(max_new_tokens):
            logits, hidden, cell = self.decoder(token_ids, hidden, cell)
            # <pad> y <sos> no son palabras que deban generarse.
            logits[:, PAD] = -torch.inf
            logits[:, SOS] = -torch.inf
            next_ids = logits.argmax(dim=-1)
            next_ids = torch.where(finished, PAD, next_ids)
            predictions.append(next_ids)
            finished = finished | (next_ids == EOS)
            token_ids = next_ids
            if finished.all().item():
                break

        return torch.stack(predictions, dim=1)


EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2

torch.manual_seed(SEED)
encoder = Encoder(len(src_vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS)
decoder = Decoder(len(tgt_vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS)
model = Seq2Seq(encoder, decoder).to(DEVICE)
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters()):,}")

### **Entrenamiento**

En entrenamiento usamos teacher forcing parcial. En validación calculamos la pérdida con teacher forcing completo: mide la predicción del siguiente token dado el prefijo correcto y permite seleccionar el modelo sin aleatoriedad en la realimentación. **No mide por sí sola la calidad de la generación libre**, que evaluaremos después.

Sumamos la pérdida de los tokens válidos y la dividimos entre su número. Así, la pérdida media no depende directamente de cuánto relleno haya en un lote. Recortamos la norma de los gradientes para limitar actualizaciones inestables.

Una época recorre todo el conjunto de entrenamiento. El tiempo depende del equipo y del tamaño elegido; empieza con menos pares para comprobar el flujo.

In [ ]:
LEARNING_RATE = 0.001
NUM_EPOCHS = 15
TEACHER_FORCING_RATIO = 0.5

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_tokens = 0
    ratio = TEACHER_FORCING_RATIO if training else 1.0

    with torch.set_grad_enabled(training):
        for source, lengths, target in loader:
            source, target = source.to(DEVICE), target.to(DEVICE)
            if training:
                optimizer.zero_grad()

            logits = model(source, lengths, target, teacher_forcing_ratio=ratio)
            expected = target[:, 1:]  # No intentamos predecir <sos>.
            loss_sum = criterion(logits.reshape(-1, logits.size(-1)), expected.reshape(-1))
            n_tokens = (expected != PAD).sum().item()
            loss = loss_sum / n_tokens

            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss_sum.item()
            total_tokens += n_tokens

    return total_loss / total_tokens


history = {"train": [], "valid": []}
best_valid_loss = math.inf
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.perf_counter()
    train_loss = run_epoch(model, train_loader, optimizer)
    valid_loss = run_epoch(model, valid_loader)
    history["train"].append(train_loss)
    history["valid"].append(valid_loss)
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = {name: tensor.detach().cpu().clone()
                      for name, tensor in model.state_dict().items()}
    print(f"Época {epoch:02d} | entrenamiento {train_loss:.3f} | "
          f"validación {valid_loss:.3f} | {time.perf_counter() - start:.1f} s")

model.load_state_dict(best_state)
model.eval()

plt.plot(range(1, NUM_EPOCHS + 1), history["train"], label="Entrenamiento (TF parcial)")
plt.plot(range(1, NUM_EPOCHS + 1), history["valid"], label="Validación (TF completo)")
plt.xlabel("Época")
plt.ylabel("Entropía cruzada media por token")
plt.legend()
plt.show()

### **Traducción**

Usamos búsqueda voraz: elegimos el token con mayor logit en cada paso. La generación termina con `<eos>` o al alcanzar el límite de seguridad. Para la comparación con atención utilizaremos exactamente el mismo procedimiento.

Una frase con palabras desconocidas pierde información antes de llegar al encoder. Conviene distinguir este problema de los errores que puede resolver la atención.


In [ ]:
@torch.no_grad()
def translate(sentence, model, max_new_tokens=50):
    model.eval()
    tokens = tokenize(sentence)
    source = torch.tensor([src_vocab.encode(tokens) + [EOS]], device=DEVICE)
    lengths = torch.tensor([source.size(1)])
    prediction = model.generate(source, lengths, max_new_tokens)
    words = tgt_vocab.decode(prediction[0].tolist())
    unknown = [token for token in tokens if token not in src_vocab.stoi]
    return " ".join(words), unknown


# Ejemplos fijos de prueba: no seleccionamos solo los que salen bien.
for en, es in test_pairs[:8]:
    prediction, unknown = translate(" ".join(en), model)
    print("EN:", " ".join(en))
    print("Referencia:", " ".join(es))
    print("Sin atención:", prediction)
    print("Palabras desconocidas:", unknown)
    print()